In [2]:
import os,ee,geemap,pandas,numpy

Map = geemap.Map(center=[29.66,116], zoom=10)
#添加面shp（文件名不能有“_”和中文）
states_shp = r'F:/file/work/师兄/extract-values/JXboundary/jxbjProject.shp'
jx = geemap.shp_to_ee(states_shp)   #必须先安装pip install pyshp #filter_polygons(states_shp)

In [3]:
Map.addLayer(jx, {}, 'JX')
#从图层中输出shp
#b = geemap.ee_to_shp(b_shp, filename=states_shp)
Map

Map(center=[29.66, 116], controls=(WidgetControl(options=['position'], widget=HBox(children=(ToggleButton(valu…

## =======================函数=======================

In [4]:
days=['31','28','31','30','31','30','31','31','30','31','30','31']
l8bands = ['B2','B3','B4','B5','B6','B7']
s2bands = ['B2','B3','B4','B8','B11','B12']
new_bands = ['B1','B2','B3','B4','B5','B6']
new_bands_ = ['B1','B2','B3','B4','B5','B6','NDVI','EVI','LSWI','NDWI','GCVI','RVI','EVI2','SAVI']
             
# 去云函数
# Function to cloud mask from the pixel_qa band of Landsat 8 SR data.
def maskL8sr(image):
    # Bits 3 and 5 are cloud shadow and cloud, respectively.
    cloudShadowBitMask = 1 << 3
    cloudsBitMask = 1 << 5
    # Get the pixel QA band.
    qa = image.select('pixel_qa')
    # Both flags should be set to zero, indicating clear conditions.
    mask = qa.bitwiseAnd(cloudShadowBitMask).eq(0) \
      .And(qa.bitwiseAnd(cloudsBitMask).eq(0))
    # Return the masked image, scaled to reflectance, without the QA bands.
    return image.updateMask(mask).divide(10000)\
      .select("B[0-9]*") \
      .copyProperties(image, ["system:time_start"]) 

def cloudfunction_L8(image):
    # image = ee.Algorithms.Landsat.simpleCloudScore(image)
    image = image.updateMask(image.select("cloud").lte(20));
    return image;

def cloudfunction_ST2(image):
    # use add the cloud likelihood band to the image
    qa = image.select("QA60")
    # Bits 10 and 11 are clouds and cirrus, respectively.
    cloudBitMask = 1 << 10;
    cirrusBitMask = 1 << 11;

    # Both flags should be set to zero, indicating clear conditions.
    mask = qa.bitwiseAnd(cloudBitMask).eq(0).And(
             qa.bitwiseAnd(cirrusBitMask).eq(0))

    # Return the masked and scaled data, without the QA bands.
    return image.updateMask(mask).divide(10000)\
        .select("B.*")\
        .copyProperties(image, ["system:index","system:time_start"])


    
def NDVI(img):
    ndvi = img.normalizedDifference(['B4', 'B3']).rename('NDVI')
    return ndvi #return img.addBands(ndvi).updateMask(mask).clipToCollection(shp)


def EVI(img):
    nir = img.select("B4")
    red = img.select("B3")
    blue = img.select("B1")
    evi = img.expression("2.5*(B4 - B3)/(B4 + 6*B3 - 7.5*B1 + 1)",{"B4": nir,"B3": red ,"B1": blue})
    evi = evi.rename('EVI')
    return evi

def LSWI(img):
    lswi = img.normalizedDifference(['B4', 'B5']).rename('LSWI')
    lswi = lswi.rename('LSWI')
    return lswi

def NDWI(img):
    ndwi = img.normalizedDifference(['B2', 'B4']).rename('NDWI')
    return ndwi

def GCVI(img):
    nir = img.select("B4")
    green = img.select("B2")
    gcvi = img.expression("B4/B2-1",{"B4": nir,"B2": green})
    gcvi = gcvi.rename('GCVI')
    return gcvi

def RVI(img):
    nir = img.select("B4")
    red = img.select("B3")
    rvi = img.expression("B4/B3",{"B4": nir,"B3": red})
    rvi = rvi.rename('RVI')
    return rvi

def EVI2(img):
    nir = img.select("B4")
    red = img.select("B3")
    evi2 = img.expression("2.5*(B4 - B3)/(B4 + 2.4*B3 + 1)",{"B4": nir,"B3": red})
    evi2 = evi2.rename('EVI2')
    return evi2

def SAVI(img):
    nir = img.select("B4")
    red = img.select("B3")
    savi = img.expression("1.5*(B4 - B3)/(B4 + B3 + 0.5)",{"B4": nir,"B3": red})
    savi = savi.rename('SAVI')
    return savi

# def NDBI(img):
#    ndbi = img.normalizedDifference(['B6', 'B5']).rename('NDBI')
#    return img.addBands(ndbi)#img.addBands(ndbi).updateMask(mask).clipToCollection(shp)


# Unions Two Images
def unionimgs1(a,b,month):
    n = len(l8bands)
    for i in range(0,n,1):
        # print(l8bands[i])
        imgA = a.select(l8bands[i])
        imgB = b.select(s2bands[i])
        if i == 0:
            mask = imgA.unmask(imgB)
        else:
            mask = mask.addBands(imgA.unmask(imgB))
    return ee.Image(mask.copyProperties(a, ["system:index","system:time_start"])).rename(new_bands)
    # return ee.Image(union.copyProperties(a, ["system:index","system:time_start"])).select(l8bands).rename(Nbands).unmask(-9999).clip(jx)

    # Unions Two Images
def unionimgs2(a,b,month):
    abands = list(a.bandNames().getInfo())# abands = geemap.image_band_names(a)# abands = list(a.getInfo().get('bands'))是错的
    bbands = list(b.bandNames().getInfo())# bbands = geemap.image_band_names(b)# bbands = list(b.getInfo().get('bands'))是错的
    # print(abands)
    # print(bbands)
    if len(abands)==len(bbands):
        n = len(abands)
        # print(abands)
        # print(bbands)
        for i in range(0,n,1):
            # print(l8bands[i])
            imgA = a.select(abands[i])
            imgB = b.select(bbands[i])
            if i == 0:
                mask = imgA.unmask(imgB)
            else:
                mask = mask.addBands(imgA.unmask(imgB))
    else:
        print("ERROR  XXXXXXXXXXXXXXXXXXXXXXXXX")
    return ee.Image(mask.copyProperties(a, ["system:index","system:time_start"])).select(abands).rename(new_bands)
    # return ee.Image(union.copyProperties(a, ["system:index","system:time_start"])).select(l8bands).rename(Nbands).unmask(-9999).clip(jx)

# ++++++++++++++++++++++采样+++++++++++++++++++++++++

In [ ]:
#参数初始化
Climit = 25 #云量
startY = 2017
endY   = 2017
startM=3
endM=11
visParams = {min: 0.2, max: 0.8, 'palette': ['blue', 'white','green']};

# 按年筛选
# imglist = [] #存放每月mosaic后的影像
img = ee.Image.constant(0).clip(jx)
for y in range(startY,startY+1,1):
    # 按月筛选
    for m in range(startM,endM+1,1):
        print(m,end='-')
        ##===================================当年===================================================================
        #更换2月天数
        if (y//4==0):
            days[2]=29
        else:
            days[2]=28
        start_date = str(y) + '-' + str(m).zfill(2) + '-01'
        end_date = str(y) + '-' + str(m).zfill(2) + '-' + str(days[m-1])
        #l8
        l8m = ee.ImageCollection('LANDSAT/LC08/C01/T1_RT_TOA')\
                .filterDate(start_date, end_date)\
                .filterBounds(jx)\
                .filterMetadata('CLOUD_COVER', 'less_than', Climit)\
                .map(ee.Algorithms.Landsat.simpleCloudScore)\
                .map(cloudfunction_L8)\
                .mosaic()#.clip(jx)#.filterMetadata('CLOUD_COVER', 'less_than', Climit)
        #S2
        s2m = ee.ImageCollection('COPERNICUS/S2')\
                        .filterDate(start_date, end_date)\
                        .filterBounds(jx)\
                        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', Climit))\
                        .map(cloudfunction_ST2)\
                        .mosaic()#.clip(jx)
        union = unionimgs1(l8m,s2m,m)
        
        ##===================================上一年===================================================================
        yy = y-1
        #更换2月天数
        if (yy//4==0):
            days[2]=29
        else:
            days[2]=28
        start_date_ = str(yy) + '-' + str(m-1).zfill(2) + '-16'
        end_date_ = str(yy) + '-' + str(m+1).zfill(2) + '-16'#+ str(days[m-1])
        
        #l8
        l8m_ = ee.ImageCollection('LANDSAT/LC08/C01/T1_RT_TOA')\
                .filterDate(start_date_, end_date_)\
                .filterBounds(jx)\
                .filterMetadata('CLOUD_COVER', 'less_than', Climit)\
                .map(ee.Algorithms.Landsat.simpleCloudScore)\
                .map(cloudfunction_L8)\
                .mosaic()#.clip(jx)#.filterMetadata('CLOUD_COVER', 'less_than', Climit)
        
        #S2
        s2m_ = ee.ImageCollection('COPERNICUS/S2')\
                        .filterDate(start_date_, end_date_)\
                        .filterBounds(jx)\
                        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', Climit))\
                        .map(cloudfunction_ST2)\
                        .mosaic()#clip(jx)#.filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
        
        union_ = unionimgs1(l8m_,s2m_,m)
        # print(list(union_.bandNames().getInfo()))
        # Map.addLayer(union_, {'bands': ['B4', 'B3', 'B2'], 'min': 0,  'max': 0.4}, 'S2_'+str(m))
        
        ##===================================下一年===================================================================
        yy = y+1
        #更换2月天数
        if (yy//4==0):
            days[2]=29
        else:
            days[2]=28
        start_date_ = str(yy) + '-' + str(m-1).zfill(2) + '-16'
        end_date_ = str(yy) + '-' + str(m+1).zfill(2) + '-16' #+ str(days[m-1])
        
        #l8
        l8m__ = ee.ImageCollection('LANDSAT/LC08/C01/T1_RT_TOA')\
                .filterDate(start_date_, end_date_)\
                .filterBounds(jx)\
                .filterMetadata('CLOUD_COVER', 'less_than', Climit)\
                .map(ee.Algorithms.Landsat.simpleCloudScore)\
                .map(cloudfunction_L8)\
                .mosaic()#.clip(jx)#.filterMetadata('CLOUD_COVER', 'less_than', Climit)
        
        #S2
        s2m__ = ee.ImageCollection('COPERNICUS/S2')\
                        .filterDate(start_date_, end_date_)\
                        .filterBounds(jx)\
                        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', Climit))\
                        .map(cloudfunction_ST2)\
                        .mosaic()#.clip(jx)#.filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
        
        union__ = unionimgs1(l8m__,s2m__,m)
        # Map.addLayer(union_, {'bands': ['B4', 'B3', 'B2'], 'min': 0,  'max': 0.4}, 'S2_'+str(m))
        
        ##===================================合并==========================================================================
        Union = unionimgs2(union,union_,m)
        Union = unionimgs2(Union,union__,m)#.unmask(-9999)#.clip(jx)
        
        # Vegetaion Index
        Union = Union.addBands( NDVI(Union))
        Union = Union.addBands( EVI(Union) )
        Union = Union.addBands( LSWI(Union))
        Union = Union.addBands( NDWI(Union))
        Union = Union.addBands( GCVI(Union))
        Union = Union.addBands( RVI(Union) )
        Union = Union.addBands( EVI2(Union))
        Union = Union.addBands( SAVI(Union))
        # print(Union.bandNames().getInfo()) 
        
        Map.addLayer(Union, {'bands': ['B3', 'B2', 'B1'], 'min': 0,  'max': 0.4}, 'U_'+str(m)) #.randomVisualizer()
        
        
        
        # ==========================Collecting monthly imageBands to 'img'==============================
        img = img.addBands(Union)
        
    # ===================Rename all bands name=======================
    newB = ['NONE']
    for x in range(startM,endM+1):
        for y in new_bands_:
            newB.append(str(x)+'M_'+y)
    img = img.rename(newB)
    # print(len(img.getInfo().get('bands')))
    # print(newB)|

Map

3-4-5-6-7-

In [12]:
# //Terrain Dem products
# //dem
dem = ee.Image('USGS/SRTMGL1_003').clip(jx)
visParam = {'min':0, 'max':3000, 'palette':["green", "blue", "red"]}
img = img.addBands(dem.select('elevation').rename('elev'))
# Map.addLayer(dem, visParam, "dem")


# //slope
slope = ee.Terrain.slope(dem).lt(10)
img = img.addBands(slope.rename('slope'))
Map.addLayer(slope, {'palette' : ["green", "blue", "red"],'min':0, 'max':9}, "slope")

EEException: Image.arrayMask: Incompatible number of bands in the mask image: 127. Expected 1.

# ++++++++++++++++++++++选取训练样本点+++++++++++++++++++++++++++

In [19]:
import pandas as pd
import os

#添加点shp
inpath = r'E:\DATA\BYLWDATA\Pt'
crop_points = inpath + os.sep + 'JXPTProj1Q.shp'
p_shp = geemap.shp_to_ee(crop_points)
Map.addLayer(p_shp,{},'sample_points') #p.addLayer(p_shp, {}, 'Points_shp')


# =====================================输出参数=====================================
outpath1 = r'E:\DATA\BYLWDATA\ExtracValue'
outpath2  = r'E:\DATA\BYLWDATA\PtAval'
outFile = outpath1 + os.sep + r'ExtractByPt1000.csv'
traPt = outpath2 + os.sep + r'ExtraAvalPt1000.shp'
# ===================================创建文件夹=====================================
if not os.path.exists(outpath1):
    os.makedirs(outpath1)
if not os.path.exists(outpath2):
    os.makedirs(outpath2)

## =============================Extract_pixel_values and Pt=========================
print('EEEEE')
geemap.extract_values_to_points(p_shp,img, outFile,scale=30)
print('END')
## Read Values And drop NaN
df = pd.read_csv(outFile,header=0)#,index_col='CID')
dfnoNan = df[~df.isin([-9999])]
dfnoNan = dfnoNan.dropna()
print(dfnoNan)

# filterMetadata()通过属性筛选有效Pt
ptl = list(dfnoNan['PtID'])
# print(ptl)
pt = p_shp.filter(ee.Filter.inList("PtID", ptl))
## Save training pt
geemap.ee_to_shp(pt, traPt)

KeyboardInterrupt: 

In [ ]:
import pandas as pd

# =====================================输出参数=====================================
outpath1 = r'E:\DATA\BYLWDATA\ExtracValue'
outpath2  = r'E:\DATA\BYLWDATA\PtAval'
outFile = outpath1 + os.sep + r'ExtractByPt1000.csv'
traPt = outpath2 + os.sep + r'ExtraAvalPt1000Proj.shp'
# ======================================读取========================================
pt = geemap.shp_to_ee(traPt)
Map.addLayer(pt,{},'sampleAva_points') #p.addLayer(p_shp, {}, 'Points_shp')
Map
## ===================================Training======================================
# This property of the table stores the land cover labels.  // 选择分类的属性
label = 'Class'
# newB.append('elev')
# newB.append('slope')

# Overlay the points on the imagery to get training.
training = img.select(newB).sampleRegions(**{
    'collection': pt,
    'properties': [label],
    'scale': 30
})

#  random uniforms to the training dataset.
withRandom = training.randomColumn() #//样本点随机的排列

# // 我们想保留一些数据进行测试，以避免模型过度拟合
split = 0.8
trainingPartition = withRandom.filter(ee.Filter.lt('random', split))#//筛选80%的样本作为训练样本
testingPartition = withRandom.filter(ee.Filter.gte('random', split))#//筛选20%的样本作为测试样本

# //分类方法选择smileCart() randomForest() minimumDistance libsvm
classifier = ee.Classifier.smileRandomForest(200).train(**{
    'features': trainingPartition,
    'classProperty': label,
    'inputProperties': newB
})

## =====================================Classify====================================

#分类
classified = img.select(newB).classify(classifier)
#图例
geemap.ee_export_image_to_drive(classified, description='Class', folder='export1Class', scale=300)
Map.addLayer(classified.randomVisualizer(), {}, 'Classification2017')
# Map.add_legend(builtin_legend='NLCD')
Map

In [ ]:
# geemap.ee_export_image(classified, filename=r'E:\DATA\BYLWDATA\Class\Class2017.tif', scale=300)
# aoi = ee.Geometry.Polygon(
#   [[[110, 32],
#     [110, 23],
#     [122,23],
#     [122, 32]]], None, False)


# function for converting GeometryCollection to Polygon/MultiPolygon
def filter_polygons(ftr):
    geometries = ftr.geometry().geometries()
    geometries = geometries.map(lambda geo: ee.Feature( ee.Geometry(geo)).set('geoType',  ee.Geometry(geo).type()))

    polygons = ee.FeatureCollection(geometries).filter(ee.Filter.eq('geoType', 'Polygon')).geometry()
    return ee.Feature(polygons).copyProperties(ftr)

# geemap.ee_export_image_to_drive(classified, description='Class', folder='exportC1',region=jx, scale=30,file_format='GeoTIFF')
# ee_export_image_to_drive(ee_object, description, folder=None, region=None, scale=None, crs=None, file_format='GeoTIFF')
jxcity = geemap.shp_to_ee('E:\DATA\BYLWDATA\Boundary\JX54.shp')   #必须先安装pip install pyshp #filter_polygons(states_shp)
Map.addLayer(jxcity, {}, 'JX')
for i in range(1,12,1):
    code = str(36)+str(i).zfill(2)
    XJ=jxcity.filterMetadata('Id','equals',int(i))
    img1 = classified.clip(XJ)
    geemap.ee_export_image_to_drive(img1, description='Class'+code, folder='exportCity1',region=XJ.geometry(), scale=30,file_format='GeoTIFF')
#     geemap.ee_export_image(img1, filename=r'E:\DATA\BYLWDATA\Class\Class'+code+r'.tif',region=XJ.geometry(), scale=30)

In [26]:


## =====================================Testing====================================
train_accuracy = classifier.confusionMatrix()
# print(train_accuracy.getInfo())                             #2D混淆矩阵
# print('总体精度   ',train_accuracy.accuracy().getInfo(),sep='=')               #总体准确性从本质上告诉我们所有参考站点中正确映射的比例。 总体准确度通常用百分比表示，其中100％的准确度是对所有参考位点都正确分类的理想分类。 总体精度是最容易计算和理解的，但最终只能为地图用户和生产者提供基本精度信息。 
# print('Kappa系数 ',train_accuracy.kappa().getInfo(),sep='=')                  #Kappa系数是通过统计测试生成的，以评估分类的准确性。 与仅随机分配的值相比，Kappa本质上评估了分类的执行情况，即分类的效果是否优于随机的值。 Kappa系数的范围为-1 t01。值为0表示分类不比随机分类好。 负数表示分类明显比随机分类差。 接近1的值表示分类明显优于随机分类。 
# print('生产者精度 ',train_accuracy.producersAccuracy().getInfo(),sep='=')      #生产者的准确性是指从地图制作者（生产者）的角度来看的地图准确性。 这是在分类地图上正确显示地面上真实特征的频率，或者是地面上某个区域的特定土地覆盖被分类的概率。 生产者的准确性是遗漏误差的补充，生产者的准确性= 100％遗漏误差。 它也是准确分类的参考位点的数量除以该类别的参考位点的总数。
# print('消费者精度 ',train_accuracy.consumersAccuracy().getInfo(),sep='=')      #消费者的准确性是指从地图用户而不是地图制作者的角度来看的准确性。 用户的准确性本质上告诉用户使用地图上的班级实际出现在地面上的频率。 这称为可靠性。 用户的准确性是佣金误差的补充，用户的准确性= 100％佣金误差。 用户的准确度是通过将特定类别的正确分类的总数除以行总数得出的。 

test = testingPartition.classify(classifier);               #//运用测试样本分类，确定要进行函数运算的数据集以及函数
test_accuracy = test.errorMatrix('Class', 'classification')
print('总体精度',test_accuracy.accuracy().getInfo())

# #     test_accuracy.kappa().getInfo()
# #     test_accuracy.producersAccuracy().getInfo()
# #     test_accuracy.consumersAccuracy().getInfo()


print('2D混淆矩阵',test_accuracy.getInfo())
print('consumers accuracy',confusionMatrix.consumersAccuracy().getInfo())#'consumers accuracy',
print('producers accuracy',confusionMatrix.producersAccuracy().getInfo());#'producers accuracy',
print('overall accuracy',confusionMatrix.accuracy().getInfo());#'overall accuracy', //面板上显示总体精度
print('kappa accuracy',confusionMatrix.kappa().getInfo());#'kappa accuracy', //面板上显示kappa值
Map

总体精度 0.7009443861490031
[[0, 0, 0, 0, 0, 0], [0, 785, 0, 0, 1, 0], [0, 0, 779, 1, 2, 1], [0, 1, 0, 756, 0, 0], [0, 0, 0, 0, 780, 3], [0, 1, 0, 1, 0, 774]]
consumers accuracy [[0, 0.676056338028169, 0.5311004784688995, 0.6846846846846847, 0.9523809523809523, 0.7639751552795031]]
producers accuracy [[0], [0.7119565217391305], [0.6733668341708543], [0.7864583333333334], [0.8756218905472637], [0.6523809523809524]]
overall accuracy 0.6971608832807571
kappa accuracy 0.6224183669594371


Map(bottom=108740.0, center=[29.6594160549124, 115.99983215332033], controls=(WidgetControl(options=['position…

In [11]:
# // Define a palette for the Land Use classification.
#//  Rice(1) #//  cultivatedland (2) #// forest(3)   #//  water (4) #// constructionland (5)  
palette = [  '0b4a8b',   '0dd66b',  '98ff00',   '62ffde',  'ff78d2'] 

# // Display the classification result and the input image.
Map.addLayer(classified, {'min': 1, 'max':6, 'palette': palette}, 'Land Use Classification')
Map

Map(bottom=7217.0, center=[26.96124577052697, 113.82934570312501], controls=(WidgetControl(options=['position'…

In [9]:
# This property of the table stores the land cover labels.
label = 'Class'

# Overlay the points on the imagery to get training.
training = img.select(newB[1:]).sampleRegions(**{
  'collection': p_shp,
  'properties': [label],
  'scale': 30
})

# Train a CART classifier with default parameters.
# trained = ee.Classifier.smileCart().train(training, label, newB)
trained = ee.Classifier.smileRandomForest().train(training, label, newB)

EEException: Required argument (numberOfTrees) missing to function: Creates an empty Random Forest classifier.

Args:
  numberOfTrees: The number of decision trees to create.
  variablesPerSplit: The number of variables per
      split. If unspecified, uses the square root of
      the number of variables.
  minLeafPopulation: Only create nodes whose training
      set contains at least this many points.
  bagFraction: The fraction of input to bag per tree.
  maxNodes: The maximum number of leaf nodes in each tree. If
      unspecified, defaults to no limit.
  seed: The randomization seed.

In [18]:
# dfnoNan = df1
# len(ptl)
# print(pt)


# # df2 = pd.read_csv(r'F:\JJUDATA\temp\pttest.shp',header=0,index_col='CID')
# # df2 = df2[newB]
# # df2
# len(newB)
l = [1,2,3,4]
l.append(1)
print(l)

[1, 2, 3, 4, 1]


In [8]:
# Classify the image with the same bands used for training.
result = img.select(newB).classify(trained)

# # Display the clusters with random colors.
Map.addLayer(result.randomVisualizer(), {}, 'classfied')
Map

Map(bottom=108740.0, center=[29.66, 116], controls=(WidgetControl(options=['position'], widget=HBox(children=(…

In [8]:
def cloudfunction_L8(image):
    # image = ee.Algorithms.Landsat.simpleCloudScore(image)
    image = image.updateMask(image.select("cloud").lte(20));
    return image;

l8bands = ['B2','B3','B4','B5','B6','B7']
#添加landsat8
l8m = ee.ImageCollection('LANDSAT/LC08/C01/T1_RT_TOA')\
                .filterDate('2017-05-01', '2017-08-01')\
                .filterBounds(jx)\
                .map(ee.Algorithms.Landsat.simpleCloudScore)\
                .map(cloudfunction_L8)\
                .mosaic()\
                .clip(jx)\
                .select(l8bands)

#添加点shp
inpath = r'F:\JJUDATA\Pt'
crop_points = inpath + os.sep + 'PtRiceProj.shp'
p_shp = geemap.shp_to_ee(crop_points)

import pandas as pd
# 输出参数
rescaled = l8m.unitScale(-2000, 10000);
outpath = r'F:\JJUDATA\ExctraValue'
outFile = outpath + os.sep + r'Ext03211119.csv'
geemap.extract_values_to_points(p_shp,rescaled, outFile)

EEException: Image.reduceRegions: The default WGS84 projection is invalid for aggregations. Specify a scale or crs & crs_transform.

In [107]:
import pandas as pd
# 输出参数
outpath = r'F:\JJUDATA\ExctraValue'
outFile = outpath + os.sep + r'ExtractByPt20210329.shp'

if not os.path.exists(outpath):
    os.makedirs(outpath)

geemap.extract_values_to_points(in_fc, dem, out_shp)
geemap.extract_values_to_points(p_shp,union,outFile)
# geemap.ee_to_shp(Ptshp,outFile)


AttributeError: module 'geemap' has no attribute 'extract_values_to_points'

In [7]:
newB = ['NONE']
for x in range(startM,endM+1):
    for y in new_bands_:
        newB.append(str(x)+'M_'+y)
img = img.rename(newB)

In [49]:
days=['31','28','31','30','31','30','31','31','30','31','30','31']
l8bands = ['B2','B3','B4','B5','B6','B7']
s2bands = ['B2','B3','B4','B8','B11','B12']
new_bands = ['B1','B2','B3','B4','B5','B6']

# 去云函数
# Function to cloud mask from the pixel_qa band of Landsat 8 SR data.
def maskL8sr(image):
    # Bits 3 and 5 are cloud shadow and cloud, respectively.
    cloudShadowBitMask = 1 << 3
    cloudsBitMask = 1 << 5
    # Get the pixel QA band.
    qa = image.select('pixel_qa')
    # Both flags should be set to zero, indicating clear conditions.
    mask = qa.bitwiseAnd(cloudShadowBitMask).eq(0) \
      .And(qa.bitwiseAnd(cloudsBitMask).eq(0))
    # Return the masked image, scaled to reflectance, without the QA bands.
    return image.updateMask(mask).divide(10000)\
      .select("B[0-9]*") \
      .copyProperties(image, ["system:time_start"]) 

def cloudfunction_L8(image):
    # image = ee.Algorithms.Landsat.simpleCloudScore(image)
    image = image.updateMask(image.select("cloud").lte(20));
    return image;

def cloudfunction_ST2(image):
    # use add the cloud likelihood band to the image
    qa = image.select("QA60")
    # Bits 10 and 11 are clouds and cirrus, respectively.
    cloudBitMask = 1 << 10;
    cirrusBitMask = 1 << 11;

    # Both flags should be set to zero, indicating clear conditions.
    mask = qa.bitwiseAnd(cloudBitMask).eq(0).And(
             qa.bitwiseAnd(cirrusBitMask).eq(0))

    # Return the masked and scaled data, without the QA bands.
    return image.updateMask(mask).divide(10000)\
        .select("B.*")\
        .copyProperties(image, ["system:index","system:time_start"])

# Unions Two Images
def unionimgs(a,b,month,union,bands):
    n = len(a.getInfo().get('bands'))
    temp = ee.Image.constant(0).clip(jx)
    for i in range(0,n,1):
        # print(i)
        imgA = a.select(l8bands[i])
        imgB = b.select(s2bands[i])
        mask = imgA.unmask(imgB)
        temp = temp.addBands(mask)
    # print(union.getInfo().get('bands'))
    Nbands = [str(month)+'m'+x for x in bands]
    temp = (ee.Image(temp.copyProperties(a, ["system:index","system:time_start"])).select(l8bands).rename(Nbands).unmask(-9999).clip(jx))
    union = union.addBands(temp)
    return union
    # return ee.Image(union.copyProperties(a, ["system:index","system:time_start"])).select(l8bands).rename(Nbands).unmask(-9999).clip(jx)
    #outimg = imgs.clip(jx)